# DAY 162 - Chatbot Basics (RAG vs Generative).
@A.IPYNB

A Large Language Model is a brilliant but pathological liar. It has memorized the internet up to a certain date, but it has no "living" memory.

If you ask it about a news event from this morning, it will either admit ignorance or, more dangerously, invent a plausible lie.

RAG (Retrieval-Augmented Generation) is the lie detector and the external brain that makes AI reliable enough for production.

### Parametric vs. Non-Parametric Memory
To understand the difference between Vanilla Generative AI and RAG, we must distinguish between two types of machine memory.

- Parametric Memory (The Generative LLM): This is the knowledge baked into the model's weights during training. It is static, expensive to update, and opaque. It represents the "Reasoning Engine."
- Non-Parametric Memory (The Retrieval System): This is an external, searchable database (like a Vector Store) that holds your proprietary or real-time data. It is dynamic, easy to update, and transparent. It represents the "Reference Library."

### The RAG Workflow
In a RAG-based chatbot, the process follows a specific lifecycle:
- The Query: The user asks a question.
- The Retrieval: The system converts the question into a mathematical vector and searches a database for the most relevant "chunks" of text.
- The Augmentation: These chunks are "stuffed" into the prompt as context.
- The Generation: The LLM reads the context and generates an answer strictly based on the provided facts.

Generative AI is a "Reasoning Engine," not a "Database." RAG allows us to decouple the logic (the LLM) from the information (the Data).

This is the professional standard for enterprise chatbots because it provides Attribution: the model can cite its sources, allowing a human to verify every claim.

In [ ]:
# @title 1. Dependencies
!pip install -q sentence-transformers transformers torch

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

print("Libraries installed. Building the brain...")

Libraries installed. Building the brain...


### The Logic: Building the Retrieval Pipeline
We use Vector Embeddings to perform semantic searches. Unlike a keyword search that looks for exact letters, a vector search looks for "Meaning" by calculating the distance between concepts in a high-dimensional space. We often use Cosine Similarity to find the closest match:$$\text{similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

In [ ]:
!pip install sentence-transformers transformers faiss-cpu torch accelerate

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import faiss
import torch

# 1. THE SPECIALIST ENGINE (Embeddings)
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 2. THE KNOWLEDGE BASE (Non-Parametric Memory)
knowledge_base = [
    "The 2026 AI Summit is held in Pokhara, Nepal on February 15th.",
    "RAG systems reduce hallucinations by 80% in clinical settings.",
    "Generative AI uses transformer architectures to predict the next token."
]
kb_embeddings = embedder.encode(knowledge_base)

# 3. THE VECTOR STORE
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(kb_embeddings))

# 4. (Generation Model)
print("Loading LLM...")
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

def generate_response(prompt, max_new_tokens=50):
    """Generate response from the LLM"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,           # beam search for more stable output
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# 5. THE TEST CASE
user_query = "Where is the 2026 AI Summit?"

# PATH A: GENERATIVE ONLY (The Guess)
print("\n" + "="*60)
print("PATH A: GENERATIVE ONLY (No RAG)")
print("="*60)
pure_prompt = f"{user_query}"
print(f"Prompt: {pure_prompt}")

pure_answer = generate_response(pure_prompt)
print(f"Answer: {pure_answer}")
print("\nWithout context, the model hallucinates!")

# PATH B: RAG AUGMENTED (The Truth)
print("\n" + "="*60)
print("PATH B: RAG AUGMENTED (With Retrieved Context)")
print("="*60)

# 6. RETRIEVAL
query_vector = embedder.encode([user_query])
distances, indices = index.search(np.array(query_vector), k=1)
retrieved_context = knowledge_base[indices[0][0]]

print(f"Retrieved Context: {retrieved_context}")
print(f"Similarity Score (L2 Distance): {distances[0][0]:.4f}\n")

# 7. GENERATION
rag_prompt = f"{retrieved_context} Q: {user_query} A:"

print(f"Prompt: {rag_prompt}")

rag_answer = generate_response(rag_prompt, max_new_tokens=30)
print(f"\nAnswer: {rag_answer}")
print("\nWith RAG, the model uses the retrieved fact!")

print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"Without RAG: {pure_answer}")
print(f"With RAG:    {rag_answer}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



PATH A: GENERATIVE ONLY (No RAG)
Prompt: Where is the 2026 AI Summit?
Answer: beijing qingdao

Without context, the model hallucinates!

PATH B: RAG AUGMENTED (With Retrieved Context)
Retrieved Context: The 2026 AI Summit is held in Pokhara, Nepal on February 15th.
Similarity Score (L2 Distance): 0.4860

Prompt: The 2026 AI Summit is held in Pokhara, Nepal on February 15th. Q: Where is the 2026 AI Summit? A:

Answer: Pokhara, Kathmandu

With RAG, the model uses the retrieved fact!

COMPARISON
Without RAG: beijing qingdao
With RAG:    Pokhara, Kathmandu


The "Chunking Strategy" is where the battle is won or lost. If you make your chunks too small, the model loses context. If you make them too large, you "pollute" the prompt with irrelevant noise. A common professional standard is 512 tokens with a 10% overlap to ensure semantic continuity.

### The Latent Space
When we "Embed" our data, we are placing it into a geometric universe. A question about "Hiking" will land geometrically close to a document about "Boots," even if the two texts share zero identical words.

1. RAG vs. Fine-Tuning
A common mistake for beginners is trying to "Fine-tune" a model to learn new facts. Specialist engineers know this is a trap.

Fine-Tuning: Best for changing the Style or Behavior (e.g., making the bot sound like a pirate).

RAG: Best for providing Facts and New Knowledge. RAG is 100x cheaper and can be updated every second by simply adding a new row to your vector database.

2. The Metric of Truth: RAGAS

In production, we don't just "feel" that a chatbot is good. We use the RAGAS framework to measure:

- Faithfulness: Does the answer actually come from the retrieved context?

- Answer Relevance: Does the answer actually address the user's question?

-  Context Precision: Was the retrieved chunk actually the best piece of information available?

### Final Verdict: Grounding the AI

Real-Time Data: RAG allows your bot to "know" things that happened five seconds ago.

Security: You can restrict the bot to only read documents the user has permission to see.

The Specialist Win: You have moved from "Chatting with a Bot" to "Engineering a Grounded System."